# HealthyAir - Feature Engineering Notebook

The original UCI file is commonly semicolon-separated, uses decimal commas, and encodes missing readings as `-200`. Check your downloaded copy. Run these notebooks in order: exploration → cleaning → feature engineering. Keep the raw file unchanged.

## Feature engineering

Use measurements available at hour *t* to predict the NO2 measurement/category at hour *t+1*. The notebook creates time and lag features. choose and cite defensible thresholds before creating categories.

In [ ]:
# Import Libraries
import sys
import pandas as pd
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import statsmodels
import scipy
import os
import gcsfs
import warnings
from scipy import stats

# Confirm location
print("Location:", sys.executable)

SEED = 42
rng = np.random.default_rng(SEED)
pd.set_option("display.max_columns", 30)
sns.set_theme(style="whitegrid", context="notebook")

print("Python:", sys.version.split()[0])
print("NumPy:", np.__version__)
print("pandas:", pd.__version__)
print("SciPy:", scipy.__version__)
print("Matplotlib:", matplotlib.__version__)
print("Seaborn:", sns.__version__)
print("statsmodels:", statsmodels.__version__)

ModuleNotFoundError: No module named 'seaborn'

## Load Data & Validate

In [ ]:
# Load data from Google Cloud Storage
GCS_PROJECT_ID = "healthy_air_analysis"
GCS_BUCKET = "uci_air_quality_20119102"
GCS_FILE = "gs://uci_air_quality_20119102/data_raw/AirQualityUCI.csv"
df = pd.read_csv(GCS_FILE)

# Data contract
required = {"Date", "Time", "CO(GT)", "PT08.S1(CO)", "NMHC(GT)", "C6H6(GT)",
            "PT08.S2(NMHC)", "NOx(GT)", "PT08.S3(NOx)", "NO2(GT)", "PT08.S4(NO2)", "PT08.S5(O3)", "T", "RH", "AH"}
assert required.issubset(df.columns)
#assert set(df["Quality"].unique()).issubset({0, 1, 3})
print("Initial data-contract checks passed.")

print("Dataset path:", GCS_BUCKET)
print(f"Rows: {df.shape[0]:,}; columns: {df.shape[1]}")
display(df.head())

df_target = df.copy()

## 1. Define the next-hour target
A simple `.shift(-1)` is only a true next-hour target if the next row is exactly one hour later. Mask gaps to avoid accidentally predicting several hours ahead.

In [ ]:
# NO2 thresholds (µg/m³)
GOOD_THRESHOLD = 100
POOR_THRESHOLD = 200

# Ensure NO2 is numeric
df_target["NO2(GT)"] = pd.to_numeric(
    df["NO2(GT)"], errors="coerce"
)

# Convert UCI missing-value marker
df_target["NO2(GT)"] = df_target["NO2(GT)"].replace(-200, np.nan)

# Create three air-quality categories
conditions = [
    df_target["NO2(GT)"] < GOOD_THRESHOLD,
    (df_target["NO2(GT)"] >= GOOD_THRESHOLD) &
    (df_target["NO2(GT)"] <= POOR_THRESHOLD),
    df_target["NO2(GT)"] > POOR_THRESHOLD
]

labels = ["GOOD", "MODERATE", "POOR"]

df_target["NO2_Category"] = np.select(
    conditions,
    labels,
    default=None
)

# Check results
print(df_target[["NO2(GT)", "NO2_Category"]].head(10))

print("\nCategory counts:")
print(df_target["NO2_Category"].value_counts(dropna=False))

print("\nCategory percentages:")
print(
    df_target["NO2_Category"]
    .value_counts(normalize=True, dropna=False)
    .mul(100)
    .round(2)
)


In [ ]:
plt.figure(figsize=(9, 5))

sns.countplot(
    data=df_target,
    x="NO2_Category",
    order=["GOOD", "MODERATE", "POOR"]
)

plt.title("Distribution of NO2 Air Quality Categories")
plt.xlabel("NO2 Category")
plt.ylabel("Number of observations")
plt.xticks(rotation=20)
plt.tight_layout()
plt.show()

In [ ]:
df_target["Target_Next_Hour"] = df_target["NO2_Category"].shift(-1)